# Eventi estremi MTGFlow 2019 — previsione diretta t+1h e t+6h

La classifica degli eventi è ricavata dallo score MTGFlow nelle sole coordinate
diurne PVGIS (`POA > 10 W/m²`). I due case study fissati sono l'episodio di
aprile 23–26 e quello di giugno 28–29, già emersi dall'analisi data-driven.

Per ogni evento si guardano le stesse metriche dei notebook precedenti, separate
per fascia di produzione: **MAE, RMSE, PICP, NMPIL, CLC**, più bias con segno,
boxplot degli errori assoluti e della NMPIL per riga, e istogrammi dell'errore.
Lo strato di riferimento è `normal`, cioè le righe non anomale dell'intero 2019.

Tutte le righe diurne valide, nessun campionamento.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.anomaly_driver as anomaly_driver
import physiq_pv.reporting.anomaly_extremes as anomaly_extremes
import physiq_pv.reporting.event_onset as event_onset
import physiq_pv.reporting.report_dump as report_dump
import physiq_pv.reporting.mtgflow_spatiotemporal as mtg_spatial

anomaly_driver = importlib.reload(anomaly_driver)
anomaly_extremes = importlib.reload(anomaly_extremes)
event_onset = importlib.reload(event_onset)
report_dump = importlib.reload(report_dump)
mtg_spatial = importlib.reload(mtg_spatial)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
FORECAST_HORIZONS = (1, 6)
DAYTIME_THRESHOLD_WM2 = 10.0
DETECTOR_SEED = 15
EVENT_WINDOWS = {
    'april_dust_23_26': ('2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26'),
    'june_extreme_28_29': ('2019-06-28', '2019-06-29'),
}

out_dir = ROOT / 'outputs' / RUN_NAME
reference_peak_by_horizon = {
    horizon: out_dir / 'posthoc_by_horizon' / f't_plus_{horizon}' / 'reference_production_peaks.csv'
    for horizon in FORECAST_HORIZONS
}
scores_path = (ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense'
               / f'seed_{DETECTOR_SEED}' / 'anomaly_scores.csv')
pvgis_2019 = Path(os.environ.get(
    'PVGIS_2019_PATH',
    '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc',
)).resolve()
labels_path = out_dir / anomaly_driver.DRIVER_LABELS_FILE

for required in (scores_path, out_dir / 'predictions.csv', pvgis_2019, *reference_peak_by_horizon.values()):
    if not required.is_file():
        raise FileNotFoundError(required)

driver_labels = (
    anomaly_driver.load_driver_labels(labels_path) if labels_path.is_file() else None
)
print('Run      :', out_dir)
print('Score    :', scores_path)
print('PVGIS    :', pvgis_2019)
print('Etichette driver:', 'caricate' if driver_labels is not None else 'assenti')

## 1. Rilevamento eventi

`EXTREME_QUANTILE` definisce cosa è "coda", `MERGE_GAP_HOURS` quanto lontani
possono stare due episodi per essere lo stesso evento. `MIN_SHARE='auto'` legge
la quota dalla distribuzione oraria: l'1% di ore diurne più intense dell'anno.
Sia il quantile sia l'aggregazione regionale escludono la notte usando la POA
PVGIS puntuale, non un intervallo orario approssimativo.

Se escono pochi eventi, abbassare `EXTREME_QUANTILE`: il taglio è un quantile
della distribuzione degli score, quindi quando gli eventi occupano una frazione
non trascurabile dell'anno il taglio finisce *dentro* la loro popolazione e ne
seleziona solo il più intenso. Con `0.99` la coda è dieci volte più larga.

In [ ]:
EXTREME_QUANTILE = 0.999
MERGE_GAP_HOURS = 48
MIN_DURATION_HOURS = 3
MAX_GAP_HOURS = 6

locations, pvgis_times, poa = mtg_spatial.load_pvgis_spatial_context(pvgis_2019)
daytime_filter = anomaly_extremes.DaytimeFilter.from_pvgis(
    locations, pvgis_times, poa, threshold_wm2=DAYTIME_THRESHOLD_WM2,
)
series = anomaly_extremes.regional_extreme_series(
    scores_path, quantile=EXTREME_QUANTILE, daytime_filter=daytime_filter,
)
episodes = anomaly_extremes.detect_extreme_episodes(
    series, min_share='auto', max_gap_hours=MAX_GAP_HOURS,
    min_duration_hours=MIN_DURATION_HOURS,
)
detected_events = anomaly_extremes.group_episodes_into_events(
    episodes, merge_gap_hours=MERGE_GAP_HOURS
)

print(f"Taglio coda      : {series.attrs['cut']:.1f}")
print(f"Quota minima     : {episodes.attrs['min_share']:.5f}")
print(f"Episodi rilevati : {len(episodes)}")
print(f"EVENTI RILEVATI  : {len(detected_events)}")
display(series['extreme_share'].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame().round(5))

if detected_events.empty:
    raise ValueError(
        'Nessun evento rilevato: abbassare EXTREME_QUANTILE (0.99) oppure '
        'MIN_DURATION_HOURS. Le celle successive non hanno nulla da confrontare.'
    )
display(detected_events.drop(columns='days'))

event_rows = []
for event_name, days in EVENT_WINDOWS.items():
    first_day = pd.Timestamp(days[0])
    last_day = pd.Timestamp(days[-1])
    window = series.loc[(series.index >= first_day) & (series.index < last_day + pd.Timedelta(days=1))]
    if window.empty:
        raise ValueError(f'Nessuna coordinata diurna MTGFlow per {event_name}.')
    peak_timestamp = window['score_max'].idxmax()
    event_rows.append({
        'event': event_name, 'start': window.index.min(), 'end': window.index.max(),
        'days': list(days),
        'duration_hours': int((window.index.max() - window.index.min()) / pd.Timedelta(hours=1)) + 1,
        'peak_timestamp': peak_timestamp, 'peak_score': float(window['score_max'].max()),
        'max_extreme_share': float(window['extreme_share'].max()),
        'n_extreme_windows': int(window['n_extreme'].sum()),
    })
events = pd.DataFrame(event_rows)
print('Case study MTGFlow selezionati, statistiche calcolate sul solo daytime:')
display(events.drop(columns='days'))

In [ ]:
# Quanti giorni copre ogni evento e con quale composizione per driver.
if not events.empty:
    summary = []
    for row in events.itertuples(index=False):
        entry = {
            'evento': row.event,
            'giorni': ' '.join(row.days),
            'n_giorni': len(row.days),
            'durata_h': row.duration_hours,
            'picco': round(row.peak_score, 1),
            'finestre_estreme': row.n_extreme_windows,
        }
        if driver_labels is not None:
            stamps = pd.to_datetime(driver_labels['timestamp'])
            window = driver_labels[
                (stamps >= pd.Timestamp(row.start).normalize())
                & (stamps <= pd.Timestamp(row.end).normalize() + pd.Timedelta(hours=23))
            ]
            mix = window['driver'].value_counts(normalize=True)
            entry.update({f'share_{name}': round(mix.get(name, 0.0), 3)
                          for name in ('solar', 'temperature', 'wind')})
            entry['driver'] = mix.idxmax() if len(mix) else ''
        summary.append(entry)
    event_summary = pd.DataFrame(summary)
    display(event_summary)

## 2. Metriche per evento e fascia di produzione

Un solo passaggio su `predictions.csv`, con ogni evento come categoria: le
statistiche sono quindi direttamente confrontabili fra eventi e con lo strato
`normal` dell'intero anno.

`EVENT_SCOPE='days'` etichetta tutte le ore dei giorni toccati dall'evento. È
voluto: lo score del detector arriva in ritardo fino alla lunghezza della
finestra (60 ore), quindi restringersi alle sole ore dell'episodio taglierebbe
parte del giorno in cui il forecaster ha davvero sofferto. Con `'hours'` si
tengono solo le ore dell'episodio.

In [ ]:
EVENT_SCOPE = 'days'   # 'days' | 'hours'

event_labels = anomaly_extremes.event_timestamp_labels(events, scope=EVENT_SCOPE)
print(f'Ore etichettate: {len(event_labels):,} su {len(events)} eventi')
print('Intervallo etichette :', event_labels['timestamp'].min(), '->',
      event_labels['timestamp'].max())

# Controllo di aggancio: le etichette devono cadere nell'anno di test della run.
_head = pd.read_csv(
    out_dir / 'predictions.csv', usecols=['timestamp', 'horizon_hours'], nrows=30000
)
_head = _head.loc[_head['horizon_hours'].isin(FORECAST_HORIZONS)]
_span = pd.to_datetime(_head['timestamp'])
print('Prime previsioni     :', _span.min(), '->', _span.max())

event_comparisons = {}
metrics_parts = []
for horizon_hours in FORECAST_HORIZONS:
    comparison = anomaly_driver.build_anomaly_driver_comparison_figures(
        out_dir, event_labels,
        figure_subdir=f'events/t_plus_{horizon_hours}/extreme_events',
        metrics_name=f'extreme_event_metrics_t_plus_{horizon_hours}.csv',
        horizon_hours=horizon_hours,
        reference_peak_path=reference_peak_by_horizon[horizon_hours],
        figures_per_category=True, chunksize=500_000,
    )
    event_comparisons[horizon_hours] = comparison
    metrics = comparison['metrics'].copy()
    metrics.insert(0, 'horizon_hours', horizon_hours)
    metrics_parts.append(metrics)
    print(f'\n=== MTGFlow t+{horizon_hours} ===')
    print('Righe per categoria:', comparison['row_counts'])
    print('CSV metriche:', comparison['metrics_path'])
event_metrics = pd.concat(metrics_parts, ignore_index=True)
display(event_metrics)

In [ ]:
for metric in ('mae', 'rmse', 'picp', 'nmpil', 'clc', 'bias'):
    print(f'\n=== {metric.upper()} ===')
    display(event_metrics.pivot(index='bin', columns=['horizon_hours', 'category'], values=metric).round(3))

In [ ]:
# Degrado relativo rispetto allo strato normale, nello stesso bin.
normal = anomaly_driver.NORMAL_CATEGORY
events_present = list(EVENT_WINDOWS)
delta_parts = []
coverage_parts = []
for horizon_hours in FORECAST_HORIZONS:
    horizon_metrics = event_metrics[event_metrics['horizon_hours'].eq(horizon_hours)]
    wide = horizon_metrics.pivot(index='bin', columns='category', values='mae')
    delta_h = (wide[events_present].div(wide[normal], axis=0) - 1.0) * 100.0
    delta_h.insert(0, 'horizon_hours', horizon_hours)
    delta_parts.append(delta_h.reset_index())
    coverage = horizon_metrics.pivot(index='bin', columns='category', values='picp')
    coverage_h = (coverage[events_present].sub(coverage[normal], axis=0)) * 100
    coverage_h.insert(0, 'horizon_hours', horizon_hours)
    coverage_parts.append(coverage_h.reset_index())
    print(f'\n=== Gap rispetto al normale, t+{horizon_hours} ===')
    display(delta_h.round(1))
    display(coverage_h.round(2))
delta = pd.concat(delta_parts, ignore_index=True)
coverage_gap = pd.concat(coverage_parts, ignore_index=True)

## 3. Figure

Per ogni fascia di produzione: boxplot Tukey esatti dell'errore assoluto e della
NMPIL per riga, istogramma dell'errore assoluto (upper 0.5% nell'ultimo bin) e
barre RMSE / PICP / CLC.

Due livelli, prodotti dalla stessa scansione:

- `figures/events/t_plus_<h>/extreme_events/` — gli eventi affiancati per orizzonte;
- la sottocartella `<evento>/` — **una cartella per evento**, con quell'evento
  accanto al solo strato `normal` e il suo `metrics.csv`. È il taglio leggibile
  quando l'evento va discusso da solo.

In [ ]:
for horizon_hours, comparison in event_comparisons.items():
    print(f'\n=== Figure combinate t+{horizon_hours} ({len(comparison["figure_paths"])}) ===')
    for key in sorted(k for k in comparison['figure_paths'] if k.startswith('driver_compare')):
        print(key)
        display(Image(filename=str(comparison['figure_paths'][key])))

In [ ]:
# Figure di un singolo evento. Cambiare SHOW_EVENT per vedere gli altri.
SHOW_EVENT = events['event'].iloc[0] if not events.empty else None

for horizon_hours, comparison in event_comparisons.items():
    folder = comparison['category_dirs'].get(SHOW_EVENT)
    if folder is None:
        print(f'Evento senza figure a t+{horizon_hours}:', SHOW_EVENT)
        continue
    print(f't+{horizon_hours}:', SHOW_EVENT, '->', folder)
    display(pd.read_csv(folder / 'metrics.csv'))
    for png in sorted(folder.glob('*.png')):
        print(png.name)
        display(Image(filename=str(png)))

## 4. Classifica: raro non vuol dire difficile

Confronto diretto fra quanto il detector considera estremo un evento (score di
picco, finestre in coda) e quanto il forecaster ne soffre davvero (degrado di
MAE e caduta di copertura, pesati sul numero di righe).

In [ ]:
rows = []
for horizon_hours in FORECAST_HORIZONS:
    horizon_metrics = event_metrics[event_metrics['horizon_hours'].eq(horizon_hours)]
    reference = horizon_metrics[horizon_metrics['category'] == normal].set_index('bin')
    for event in events_present:
        subset = horizon_metrics[horizon_metrics['category'] == event]
        weights = subset['count'].to_numpy(float)
        if weights.sum() == 0:
            continue
        aligned = reference.loc[subset['bin']]
        rows.append({
            'horizon_hours': horizon_hours, 'evento': event,
            'n_righe': int(weights.sum()),
            'mae': float(np.average(subset['mae'], weights=weights)),
            'mae_normale': float(np.average(aligned['mae'], weights=weights)),
            'picp': float(np.average(subset['picp'], weights=weights)),
            'picp_normale': float(np.average(aligned['picp'], weights=weights)),
            'clc': float(np.average(subset['clc'], weights=weights)),
        })

ranking = pd.DataFrame(rows)
if ranking.empty:
    print('Nessun evento con righe diurne valide: classifica non calcolabile.')
else:
    ranking['mae_gap_%'] = ((ranking['mae'] / ranking['mae_normale'] - 1) * 100).round(1)
    ranking['picp_gap_pt'] = ((ranking['picp'] - ranking['picp_normale']) * 100).round(2)
    ranking = ranking.merge(
        events[['event', 'peak_score', 'n_extreme_windows']],
        left_on='evento', right_on='event', how='left',
    ).drop(columns='event').sort_values(['horizon_hours', 'mae_gap_%'], ascending=[True, False])
    display(ranking.round(3))

In [ ]:
# Con due case study la correlazione di rango non è informativa: confronto diretto t+1/t+6.
if not ranking.empty:
    display(ranking.pivot(index='evento', columns='horizon_hours', values=[
        'mae', 'mae_gap_%', 'picp', 'picp_gap_pt',
    ]).round(3))

## 5. Il modello insegue l'evento o lo vede arrivare?

Il forecaster non riceve covariate all'ora che deve prevedere: la finestra di
input si chiude prima. All'arrivo della nube le ultime ore di input descrivono
ancora cielo sereno, quindi un modello puramente reattivo continuerebbe a
prevedere produzione normale e si adatterebbe solo quando il crollo entra nella
finestra.

Il test: per ogni nodo si individua **l'ora di arrivo** dell'evento (prima ora in
cui la produzione scende sotto `1 - SHORTFALL` del riferimento dello stesso nodo
alla stessa ora nei giorni tranquilli precedenti), poi si media l'errore per
**ore trascorse dall'arrivo**.

- bias grande e positivo a lag 0–2 che decade verso zero → il modello **insegue**;
- bias piatto a tutti i lag → il modello **vede** l'evento nei suoi input.

L'arrivo è calcolato per nodo, non per la regione: la nube non tocca tutte le
località alla stessa ora.

In [ ]:
ONSET_EVENT = 'april_dust_23_26'
ONSET_DAYS = ['2019-04-24']     # giorni dell'evento
EXCLUDE_DAYS = ['2019-04-22', '2019-04-23']   # salita: fuori dalla baseline
BASELINE_DAYS = 12
SHORTFALL = 0.4                 # calo minimo per dire "l'evento e arrivato"
MAX_LAG = 10

onset_responses = {}
for horizon_hours in FORECAST_HORIZONS:
    window = event_onset.load_event_window(
        out_dir, event_days=ONSET_DAYS, baseline_days=BASELINE_DAYS,
        exclude_days=EXCLUDE_DAYS, horizon_hours=horizon_hours,
    )
    response = event_onset.build_onset_response(
        window, shortfall=SHORTFALL, max_lag_hours=MAX_LAG,
    )
    onset_responses[horizon_hours] = response
    print(f'\n=== Arrivo evento t+{horizon_hours} ===')
    print(f"Nodi raggiunti: {response['n_nodes_reached']}; mai raggiunti: {response['unaffected']}")
    display(response['profile'].round(3))

In [ ]:
onset_paths = {}
for horizon_hours, response in onset_responses.items():
    fig = event_onset.plot_onset_response(
        response, title=f'Risposta del forecaster t+{horizon_hours}h — evento {ONSET_EVENT}'
    )
    onset_dir = out_dir / 'figures' / 'events' / f't_plus_{horizon_hours}' / 'extreme_events' / ONSET_EVENT
    onset_dir.mkdir(parents=True, exist_ok=True)
    onset_path = onset_dir / 'onset_response.png'
    fig.savefig(onset_path, dpi=140, bbox_inches='tight')
    plt.close(fig)
    onset_paths[horizon_hours] = onset_path
    display(Image(filename=str(onset_path)))

In [ ]:
# Lettura sintetica: quanto dura il ritardo prima che il bias rientri.
onset_summary = []
for horizon_hours, response in onset_responses.items():
    profile = response['profile']
    if profile.empty:
        continue
    peak = profile.loc[profile['bias'].idxmax()]
    settled = profile.loc[profile['bias'].abs() < 0.25 * abs(peak['bias'])]
    onset_summary.append({
        'horizon_hours': horizon_hours, 'peak_bias': peak['bias'],
        'peak_lag_hours': int(peak['lag_hours']),
        'settled_lag_hours': int(settled['lag_hours'].min()) if not settled.empty else np.nan,
    })
display(pd.DataFrame(onset_summary).round(3))

## 6. Riepilogo da copiare

In [ ]:
report_dump.dump_sections({
    'eventi': events.drop(columns='days') if not events.empty else None,
    'eventi_riassunto': event_summary if 'event_summary' in dir() else None,
    'righe_per_categoria': pd.DataFrame([
        {'horizon_hours': horizon, **comparison['row_counts']}
        for horizon, comparison in event_comparisons.items()
    ]),
    'metriche_evento': event_metrics,
    'delta_mae_vs_normale': delta if 'delta' in dir() else None,
    'riepilogo_arrivo_evento': pd.DataFrame(onset_summary),
    'classifica': ranking if 'ranking' in dir() and not ranking.empty else None,
}, path=out_dir / 'analysis_summary_extreme_events_t1_t6.txt', max_rows=120)